In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
!uv pip install lightgbm scikit-learn pandas pyarrow matplotlib ucimlrepo --quiet

In [ ]:
import lightgbm as lgbm
from pomap.interface import leaf, feed, ensemble, split

import polars as pl

from functools import partial
from ucimlrepo import fetch_ucirepo

In [ ]:
dataset = fetch_ucirepo(id=601)
data = pl.from_pandas(dataset.data.original)
features = dataset.data.features.columns
target = "Machine failure"

In [ ]:
# Convert String features to categoricals so that LGBM doesn't complain
for feature in data.columns:
    if data.schema[feature] == pl.String:
        data = data.with_columns(pl.col(feature).cast(pl.Enum(data[feature].unique())))

In [ ]:
# Give ourselves a row index to keep track of train and test splits
data = data.with_row_index(name="row")

In [ ]:
class LGBMWrapper:
    def __init__(self, features, target, task_type, **kwargs):
        if task_type == "regression":
            self.model = lgbm.LGBMRegressor(**kwargs)
        elif task_type == "classification":
            self.model = lgbm.LGBMClassifier(**kwargs)
        else:
            raise ValueError(f"Unknown task type {task_type}")

        self.features = features
        self.target = target

    def fit(self, training_set: pl.DataFrame, validation_set: pl.DataFrame = None):

        X_train = training_set.select(*self.features).to_pandas()
        y_train = training_set[self.target].to_pandas()

        eval_set = [(X_train, y_train)]
        eval_names = ["training"]

        if validation_set is not None:
            X_val = validation_set.select(*self.features).to_pandas()
            y_val = validation_set[self.target].to_pandas()
            eval_set += [(X_val, y_val)]
            eval_names += ["validation"]

        self.model.fit(X_train, y_train, eval_set=eval_set, eval_names=eval_names)

    def predict(self, df: pl.DataFrame):
        X = df.select(*self.features).to_pandas()
        return self.model.predict(X)

In [ ]:
teacher_hyperparams = dict(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    num_leaves=63,
)

student_hyperparams = dict(
    n_estimators=25,
    max_depth=3,
    learning_rate=0.05,
    num_leaves=15,
)

teacher_model = partial(
    LGBMWrapper,
    task_type="classification",
    features=features,
    target=target,
    **teacher_hyperparams,
)

student_model = partial(
    LGBMWrapper,
    task_type="regression",
    features=features,
    target="teacher",
    **student_hyperparams,
)

untutored_student = partial(
    LGBMWrapper,
    task_type="classification",
    features=features,
    target=target,
    **student_hyperparams,
)

In [ ]:
teacher = leaf(teacher_model, "teacher")
student = leaf(student_model, "student")
untutored = leaf(untutored_student, "untutored_student")

In [ ]:
distillation = feed("distillation", source=teacher, consumer=student)
all_models = ensemble("full", distillation, untutored)

In [ ]:
full_experiment = split(
    name="train_test_split",
    model=all_models,
    train_filter=pl.col("row") < pl.col("row").max() * 0.75,
    test_filter=pl.lit(True),  # Generate a prediction on every row so we can see
)

In [ ]:
full_experiment.mark_train_validation_test_rows(data).to_pandas().set_index(
    "row"
).filter(regex="__").cumsum().plot()

In [ ]:
full_experiment.fit(data);

In [ ]:
fitted = full_experiment.predict(data)
fitted = fitted.with_columns(student_predictions=pl.col("student").clip(0, 1))

In [ ]:
full_experiment.mark

In [ ]:
import polars.selectors as cs

fitted.with_columns(
    *[
        (pl.col(c) - pl.col(target)).alias(f"residual_{c}")
        for c in ["teacher", "student_predictions", "untutored_student"]
    ]
).select(cs.contains("residual").pow(2).mean())